# Exploratory Data Analysis (EDA) on Car Evaluation Dataset
### F.Y. BCA Semester I (Division B) — Group Project

- **Group Number:** Group 6
- **Roll Numbers:** 33, 42, 35, 36, 37, 38, 103, 34
- **Dataset:** UCI Machine Learning Repository — Car Evaluation Dataset (1,728 records)
- **Submission Deadline:** 5th October 2026

---
**Note:** This notebook runs easily both locally and in Google Colab. If running on Google Colab, it will automatically load the dataset from the official UCI website.

## 1. Introduction & Objective

### Introduction
When people want to buy a car, they look at several factors such as the purchase price, maintenance costs, how many people it can carry, and how safe it is. Different cars offer different combinations of these features.

### Objective
The objective of our project is to perform **Exploratory Data Analysis (EDA)** on 1,728 car records. We want to find out:
1. What are the common characteristics of cars in this dataset?
2. Which features have the biggest impact on whether a car is evaluated as **unacceptable (`unacc`)**, **acceptable (`acc`)**, **good (`good`)**, or **very good (`vgood`)**?
3. Are there any dealbreakers (like low safety or small capacity) that cause a car to be rejected immediately?

## 2. Dataset Description

The **Car Evaluation Dataset** was downloaded from the **UCI Machine Learning Repository**.
- It was created by researchers Marko Bohanec and Vladislav Rajkovič (1990).
- **Total rows (cars):** 1,728
- **Total columns:** 7 (6 input specifications + 1 evaluation class)
- **Design:** The dataset covers every possible combination of the 6 features ($4 \times 4 \times 4 \times 3 \times 3 \times 3 = 1,728$).
- **Missing data:** There are zero missing values.

## 3. Data Types and Variables

All 7 columns in this dataset contain categorical data (categories with a natural order, called ordinal variables):

| Variable Name | Role | Options | Description |
| :--- | :--- | :--- | :--- |
| `buying` | Input Feature | `vhigh`, `high`, `med`, `low` | Initial purchase price tier |
| `maint` | Input Feature | `vhigh`, `high`, `med`, `low` | Maintenance and servicing expense |
| `doors` | Input Feature | `2`, `3`, `4`, `5more` | Number of doors |
| `persons` | Input Feature | `2`, `4`, `more` | Seating capacity in persons |
| `lug_boot` | Input Feature | `small`, `med`, `big` | Size of the luggage boot |
| `safety` | Input Feature | `low`, `med`, `high` | Safety rating of the vehicle |
| **`class`** | **Target Variable** | `unacc`, `acc`, `good`, `vgood` | Overall car acceptability |

## 4. Tools Used for Analysis

We used the standard Python data science libraries for this project:
- **Python:** Core programming language.
- **Pandas:** For loading and manipulating tabular data.
- **NumPy:** For numerical operations.
- **Matplotlib & Seaborn:** For drawing charts and heatmaps.
- **Jupyter Notebook / Google Colab:** For running the code step-by-step.
- **GitHub:** For version control and sharing the project code.

In [ ]:
# Step 4.1: Import the libraries we need
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110

# Consistent colors for each class: Red for unacceptable, Orange for acceptable, Blue for good, Green for very good
class_colors = {
    'unacc': '#e74c3c',
    'acc': '#f39c12',
    'good': '#3498db',
    'vgood': '#2ecc71'
}
class_order = ['unacc', 'acc', 'good', 'vgood']

print('Libraries imported successfully!')

In [ ]:
# Step 4.2: Load the car evaluation data
col_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']

local_file = 'data/car.data'
online_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'

if os.path.exists(local_file):
    df = pd.read_csv(local_file, names=col_names)
    print('Loaded data from local folder:', local_file)
else:
    df = pd.read_csv(online_url, names=col_names)
    print('Loaded data directly from UCI online repository')

print('Total rows and columns:', df.shape)
df.head()

## 5. Data Quality Assessment

Before analyzing the data, we check if there are any missing values, empty cells, or duplicate rows.

In [ ]:
# Check for missing/null values
print('--- Missing Values Check ---')
print(df.isnull().sum())

# Check for duplicate rows
print('\nTotal duplicate rows:', df.duplicated().sum())

**Observation:** The dataset is completely clean. There are **0 missing values** and **0 duplicate rows**, so no rows needed to be dropped or cleaned.

## 6. Descriptive Statistics

Because all columns contain text categories rather than numbers, we look at frequency counts (how often each option appears) and percentages.

In [ ]:
# Summary of target class frequencies
class_counts = df['class'].value_counts()[class_order]
class_percent = (class_counts / len(df) * 100).round(2)

summary_table = pd.DataFrame({
    'Number of Cars': class_counts,
    'Percentage (%)': class_percent
})
summary_table

**Observation:** Over **70% of the cars are unacceptable (`unacc`)**. Only 22.2% are acceptable, and less than 4% reach 'good' or 'very good'. This shows that the standards used to evaluate these cars are quite strict.

## 7. Univariate Analysis

Univariate analysis means looking at **one feature at a time**. Below, we plot the distribution of our target variable (`class`) and the 6 input features.

In [ ]:
# Chart: Distribution of the target variable (class)
fig, ax = plt.subplots(figsize=(8, 4.5))

sns.barplot(
    x=class_counts.index,
    y=class_counts.values,
    palette=class_colors,
    ax=ax,
    edgecolor='black'
)

for i, count in enumerate(class_counts.values):
    pct = (count / len(df)) * 100
    ax.text(i, count + 20, f'{count} ({pct:.1f}%)', ha='center', fontweight='bold')

ax.set_title('Target Class Distribution (Car Acceptability)', fontsize=13, fontweight='bold')
ax.set_xlabel('Evaluation Category')
ax.set_ylabel('Number of Cars')
ax.set_ylim(0, 1400)
plt.tight_layout()
plt.show()

In [ ]:
# Chart: Distribution of the 6 input features
features = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for idx, feat in enumerate(features):
    ax = axes[idx // 3, idx % 3]
    counts = df[feat].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color='#34495e', edgecolor='black')
    ax.set_title(f'Feature: {feat}', fontsize=11, fontweight='bold')
    ax.set_xlabel(feat.capitalize())
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v/2, f'{v}', ha='center', va='center', color='white', fontweight='bold')

plt.suptitle('Distribution of Input Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observation:** Notice that every input feature has an equal count across its options (for example, exactly 432 cars per price tier, and 576 cars per safety tier). This is because the dataset was created using a complete factorial grid.

## 8. Bivariate Analysis

Bivariate analysis means looking at **two features together**. Here, we compare our input features against the target `class` to see how each feature influences whether a car is accepted or rejected.

In [ ]:
# Bivariate 1: Safety vs Acceptability
cross_safety = pd.crosstab(df['safety'], df['class'])[class_order]
cross_safety_pct = pd.crosstab(df['safety'], df['class'], normalize='index')[class_order] * 100

cross_safety_pct.plot(
    kind='bar',
    stacked=True,
    color=[class_colors[c] for c in class_order],
    edgecolor='black',
    figsize=(8, 4.5)
)
plt.title('Safety Rating vs. Car Acceptability (%)', fontsize=13, fontweight='bold')
plt.xlabel('Safety Rating')
plt.ylabel('Percentage within Tier (%)')
plt.legend(title='Class', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Cross-tabulation count table:')
display(cross_safety)

**Key Finding on Safety:** Look at the 'low' safety row. **All 576 cars with low safety are unacceptable (`unacc`)**. There are zero accepted cars. This proves that safety is a strict requirement—no one accepts a car with poor safety.

In [ ]:
# Bivariate 2: Passenger Capacity vs Acceptability
cross_persons = pd.crosstab(df['persons'], df['class'])[class_order]
cross_persons_pct = pd.crosstab(df['persons'], df['class'], normalize='index')[class_order] * 100

cross_persons_pct.plot(
    kind='bar',
    stacked=True,
    color=[class_colors[c] for c in class_order],
    edgecolor='black',
    figsize=(8, 4.5)
)
plt.title('Passenger Capacity vs. Car Acceptability (%)', fontsize=13, fontweight='bold')
plt.xlabel('Seating Capacity')
plt.ylabel('Percentage within Tier (%)')
plt.legend(title='Class', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Cross-tabulation count table:')
display(cross_persons)

**Key Finding on Capacity:** All 576 2-seater cars are unacceptable. To be accepted, a car must seat at least 4 passengers.

In [ ]:
# Bivariate 3: Price & Maintenance vs Acceptability
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

cross_buy = pd.crosstab(df['buying'], df['class'], normalize='index')[class_order] * 100
cross_buy.plot(kind='bar', stacked=True, color=[class_colors[c] for c in class_order], ax=axes[0], edgecolor='black')
axes[0].set_title('Buying Price vs Acceptability (%)', fontweight='bold')
axes[0].set_xlabel('Buying Price')
axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=0)

cross_maint = pd.crosstab(df['maint'], df['class'], normalize='index')[class_order] * 100
cross_maint.plot(kind='bar', stacked=True, color=[class_colors[c] for c in class_order], ax=axes[1], edgecolor='black')
axes[1].set_title('Maintenance Cost vs Acceptability (%)', fontweight='bold')
axes[1].set_xlabel('Maintenance Cost')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Key Finding on Price:** When the buying price or maintenance is 'vhigh' (very high), a car **never achieves 'good' or 'very good'**. High costs cap the rating.

## 9. Grouping and Aggregation

Here we combine **Safety** and **Passenger Capacity** together to see what happens when both features interact.

In [ ]:
# Grouping by Safety and Capacity
grouped = df.groupby(['safety', 'persons', 'class'], observed=False).size().unstack(fill_value=0)
grouped['Total'] = grouped.sum(axis=1)
grouped['Acceptable or Better'] = grouped['acc'] + grouped['good'] + grouped['vgood']
grouped['Acceptance Rate (%)'] = (grouped['Acceptable or Better'] / grouped['Total'] * 100).round(1)
grouped[['unacc', 'acc', 'good', 'vgood', 'Total', 'Acceptance Rate (%)']]

**Observation:** When safety is high AND capacity is 4 or more, the acceptance rate jumps to over **74% to 78%**. This shows how fulfilling both baseline rules makes a car attractive to buyers.

## 10. Correlation Analysis

Because our features have logical ranks (like low, medium, high), we convert them into numbers from 1 to 4 and calculate the **Spearman Rank Correlation**.

In [ ]:
# Convert categories into logical ranks
rank_map = {
    'buying': {'low': 1, 'med': 2, 'high': 3, 'vhigh': 4},
    'maint': {'low': 1, 'med': 2, 'high': 3, 'vhigh': 4},
    'doors': {'2': 2, '3': 3, '4': 4, '5more': 5},
    'persons': {'2': 2, '4': 4, 'more': 6},
    'lug_boot': {'small': 1, 'med': 2, 'big': 3},
    'safety': {'low': 1, 'med': 2, 'high': 3},
    'class': {'unacc': 0, 'acc': 1, 'good': 2, 'vgood': 3}
}

df_numeric = df.copy()
for col in rank_map:
    df_numeric[col] = df_numeric[col].map(rank_map[col])

# Correlation heatmap
plt.figure(figsize=(7.5, 5.5))
sns.heatmap(df_numeric.corr(method='spearman'), annot=True, fmt='.2f', cmap='Blues', linewidths=1)
plt.title('Spearman Rank Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:**
- **Safety (+0.44)** and **Persons (+0.34)** have the highest positive correlation with car acceptability.
- **Buying Price (-0.28)** and **Maintenance (-0.23)** have negative correlations, showing that higher costs lower the rating.

## 11. Visualization

During this project, we generated 7 high-resolution charts saved in the `figures/` folder. Each chart uses consistent color coding so that findings are easy to read and understand.

## 12. Key Findings

From our analysis, we identified 5 clear rules that govern car evaluations:
1. **Safety is Mandatory:** 100% of cars with low safety are unacceptable.
2. **2-Seater Penalty:** 100% of 2-seater cars are rejected as unacceptable.
3. **Price Limits Top Ratings:** Very high buying price or maintenance prevents cars from being rated 'good' or 'very good'.
4. **Requirements for 'Very Good':** Every 'very good' car has high safety, accommodates 4+ persons, and has low/medium costs.
5. **Doors and Luggage are Secondary:** Doors and luggage boot size only help choose between 'acc' and 'good' once safety and capacity pass.

## 13. Conclusion

Exploratory Data Analysis showed that car evaluation follows a simple step-by-step logic:
- **Step 1 (Safety & Seats):** If a car fails on safety or only has 2 seats, it is rejected immediately.
- **Step 2 (Affordability):** If safety and capacity pass, affordability decides whether it is acceptable.
- **Step 3 (Extra Comfort):** Large luggage space and cheap maintenance elevate an acceptable car into 'good' or 'very good'.

## 14. References

1. UCI Machine Learning Repository — Car Evaluation Dataset: https://archive.ics.uci.edu/dataset/19/car+evaluation
2. Bohanec, M., & Rajkovič, V. (1990). *Expert system for decision making*. Sistemica, 1(1), 145-157.
3. McKinney, W. *Python for Data Analysis*. O'Reilly Media.
4. Pandas & Matplotlib Official Documentation.

## 15. Code (GitHub)

- **GitHub Repository:** https://github.com/Parth-ux-arch/Eda-Data_analysis-Project.git
- Contains the complete dataset, charts, and this Jupyter Notebook.